# Importing Graph and creating Laplacain Matrix

In [ ]:
import numpy as np
import math

file_path = "P2P-Gnutella.txt"
# Load edges from file (assuming whitespace or comma separated: nodeA nodeB per line)
edges = np.loadtxt(file_path, dtype=int)

# Find the number of nodes (assuming nodes are labeled from 0 to N-1, adjust if necessary)
n_nodes = edges.max() + 1
print(f"n_nodes:{n_nodes}")

# Initialize adjacency matrix
adj = np.zeros((n_nodes, n_nodes), dtype=int)

# Fill adjacency matrix (assuming undirected and unweighted)
for a, b in edges:
    adj[a, b] = 1
    adj[b, a] = 1

In [ ]:
# Degree matrix: diagonal with node degrees
degrees = adj.sum(axis=1)
deg_matrix = np.diag(degrees)

# Laplacian: L = D - A
laplacian = deg_matrix - adj

# Calculating all pairs effective resistance matrix

In [ ]:
def effective_resistance_all_pairs(L):
    # Compute Moore-Penrose pseudoinverse of Laplacian
    L_pinv = np.linalg.pinv(L)

    # Diagonal elements of pseudoinverse
    diag = np.diag(L_pinv)

    # Compute effective resistance matrix R
    # R[i,j] = L^+_ii + L^+_jj - 2 * L^+_ij
    R = diag[:, np.newaxis] + diag[np.newaxis, :] - 2 * L_pinv
    return R

In [ ]:
R = effective_resistance_all_pairs(laplacian)
kirchoff_at_start = R.sum()/2
print(kirchoff_at_start/math.comb(n_nodes,2))
# print(R)

#Number of edges to be added

In [ ]:
k = 50

# Greedy Algorithm


In [ ]:
import numpy as np
from tqdm import tqdm

def update_pseudoinverse(L_pinv, u):
    # Sherman-Morrison update for pseudoinverse
    Lp_u = L_pinv @ u
    denom = 1 + u @ Lp_u
    return L_pinv - np.outer(Lp_u, Lp_u) / denom

def update_pseudoinverse_square(L_pinv, L2_pinv, u):
    Lp_u = L_pinv @ u            # Vector
    L2_u = L2_pinv @ u           # Vector
    denom = 1 + u.T @ Lp_u       # Scalar
    uTL2u = u.T @ L2_u           # Scalar

    term1 = np.outer(Lp_u, L2_u) + np.outer(L2_u, Lp_u)   # Symmetric term
    term2 = uTL2u * np.outer(Lp_u, Lp_u)                 # Outer product scaled

    return L2_pinv - term1 / denom + term2 / (denom**2)



def greedy_add_k_edges_kirchhoff_fast(L, k):
    n = L.shape[0]
    L_current = L.copy()
    L_pinv = np.linalg.pinv(L_current)
    L2_pinv = L_pinv @ L_pinv
    added_edges = []
    for step in range(k):
        adj = -np.minimum(L_current, 0)
        connected = adj > 0
        candidates = [(i, j) for i in range(n) for j in range(i + 1, n) if not connected[i, j]]
        # print(candidates)
        if not candidates:
            break
        best_improvement = None
        best_edge = None
        for i, j in tqdm(candidates):
            denom = 1 + (L_pinv[i, i] - 2 * L_pinv[i, j] + L_pinv[j, j])
            num = L2_pinv[i, i] - L2_pinv[i, j] - L2_pinv[j, i] + L2_pinv[j, j]
            improvement = num / denom
            # print(i,j,improvement)
            if best_improvement is None or improvement > best_improvement:
                best_improvement = improvement
                best_edge = (i, j)
        if best_edge is None:
            break
        i, j = best_edge
        u = np.zeros(n)
        u[i] = 1
        u[j] = -1
        L_current = L_current + np.outer(u, u)
        # Update pseudoinverse and its square using Sherman-Morrison
        L2_pinv = update_pseudoinverse_square(L_pinv, L2_pinv, u)
        L_pinv = update_pseudoinverse(L_pinv, u)
        if((step+1)%5 == 0):
          print("Kirchoff index after "+ str(step+1)+" edge additions:",n*(np.trace(L_pinv)/math.comb(n_nodes,2)))

        added_edges.append(best_edge)
        # print(best_edge)
    diag = np.diag(L_pinv)
    R_new = diag[:, np.newaxis] + diag[np.newaxis, :] - 2 * L_pinv
    return added_edges, L_current, R_new

In [ ]:
greedy_edges_added, greedy_L_updated, greedy_resistance_matrix = greedy_add_k_edges_kirchhoff_fast(laplacian,k)